# MATH 5010 Computer Lab — Section 6  
## Conditional Distributions and Conditional Expectations  
### Full Solutions Included

**Main ideas.** This lab turns the Section 6 lecture material into computational practice.  
We will use Python to study:

1. Conditional distributions  
2. Law of total probability  
3. Best-prize/secretary problem  
4. Gambler's ruin  
5. Memoryless property of the exponential distribution  
6. Conditional expectation as a random variable  
7. Law of total expectation and law of total variance  
8. Random sums  
9. Covariance identity  
10. Bayesian updating  
11. Missing data and inverse probability weighting  
12. Survey sampling ideas

All exercises include full solutions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import comb, log, sqrt
from scipy import stats
from scipy.special import gammaln

rng = np.random.default_rng(5010)

pd.set_option("display.precision", 4)
print("Packages loaded.")

## 1. Conditional Distributions from a Joint PMF

Suppose $(X,Y)$ has the following joint probability mass function.

\[
p_{X,Y}(x,y)=P(X=x,Y=y).
\]

The conditional PMF is

\[
p_{X\mid Y}(x\mid y)=\frac{p_{X,Y}(x,y)}{p_Y(y)}.
\]

The law of total probability says

\[
P(X=x)=\sum_y P(X=x\mid Y=y)P(Y=y).
\]

In [ ]:
# A small joint PMF table.
# Rows are x-values, columns are y-values.
x_values = np.array([0, 1, 2])
y_values = np.array([0, 1])

joint = pd.DataFrame(
    [[0.10, 0.05],
     [0.20, 0.25],
     [0.15, 0.25]],
    index=x_values,
    columns=y_values
)

joint.index.name = "x"
joint.columns.name = "y"

print("Joint PMF:")
display(joint)

print("Total probability:", joint.to_numpy().sum())

In [ ]:
# Marginals
pX = joint.sum(axis=1)
pY = joint.sum(axis=0)

print("Marginal PMF of X:")
display(pX.to_frame("P(X=x)"))

print("Marginal PMF of Y:")
display(pY.to_frame("P(Y=y)"))

In [ ]:
# Conditional PMF of X given Y=y: divide each column by its column sum
pX_given_Y = joint.div(pY, axis=1)

print("Conditional PMF P(X=x | Y=y):")
display(pX_given_Y)

# Conditional PMF of Y given X=x: divide each row by its row sum
pY_given_X = joint.div(pX, axis=0)

print("Conditional PMF P(Y=y | X=x):")
display(pY_given_X)

### Solution Check: Law of Total Probability

We recover $P(X=x)$ from the conditional probabilities:

\[
P(X=x)=\sum_y P(X=x\mid Y=y)P(Y=y).
\]

In [ ]:
pX_reconstructed = pX_given_Y.mul(pY, axis=1).sum(axis=1)
check = pd.DataFrame({
    "Direct P(X=x)": pX,
    "From total probability": pX_reconstructed
})
display(check)

print("All close?", np.allclose(pX.values, pX_reconstructed.values))

## 2. Conditional Probability with a Continuous Conditioning Variable

The lecture emphasizes the identity

\[
P(A)=\int_{-\infty}^{\infty}P(A\mid X=x)f_X(x)\,dx
=E[P(A\mid X)].
\]

We will verify this by simulation.

Let $X\sim \text{Uniform}(0,1)$ and, conditional on $X=x$, let

\[
Y\mid X=x\sim \text{Bernoulli}(x).
\]

Then

\[
P(Y=1)=E[P(Y=1\mid X)]=E[X]=\frac12.
\]

In [ ]:
N = 200_000
X = rng.uniform(0, 1, size=N)
Y = rng.binomial(1, X)

estimated_prob = Y.mean()
theory_prob = 0.5

print("Estimated P(Y=1):", estimated_prob)
print("Theoretical P(Y=1):", theory_prob)

### Full Solution

Since $P(Y=1\mid X=x)=x$,

\[
P(Y=1)=E[P(Y=1\mid X)]=E[X]=\int_0^1 x\,dx=\frac12.
\]

The simulation should be close to $0.5$.

## 3. Best-Prize Problem

There are $n$ prizes arriving in random order. The strategy is:

1. Reject the first $k$ prizes.
2. Then accept the first prize that is better than all previous prizes.

For this strategy, an approximation from the lecture is

\[
P_k(\text{best})\approx \frac{k}{n}\log\left(\frac{n}{k}\right).
\]

More exactly, for $k\ge 1$,

\[
P_k(\text{best})=\sum_{i=k+1}^n \frac{1}{n}\frac{k}{i-1}.
\]

In [ ]:
def secretary_success_probability(n, k):
    if k == 0:
        return 1/n
    return sum((1/n) * (k/(i-1)) for i in range(k+1, n+1))

n = 100
ks = np.arange(0, n)
exact_probs = np.array([secretary_success_probability(n, k) for k in ks])

best_k = ks[np.argmax(exact_probs)]
best_prob = exact_probs.max()

print(f"n = {n}")
print(f"Best k = {best_k}")
print(f"Best exact success probability = {best_prob:.4f}")
print(f"n/e ≈ {n/np.e:.2f}")
print(f"1/e ≈ {1/np.e:.4f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(ks, exact_probs)
plt.axvline(best_k, linestyle="--", label=f"best k = {best_k}")
plt.xlabel("k: number of prizes rejected first")
plt.ylabel("Success probability")
plt.title("Best-Prize Problem: Success Probability by k")
plt.legend()
plt.show()

In [ ]:
def simulate_secretary(n, k, reps=50_000):
    successes = 0
    for _ in range(reps):
        # Larger value means better prize
        order = rng.permutation(np.arange(1, n+1))
        best_value = n

        if k == 0:
            chosen = order[0]
        else:
            benchmark = order[:k].max()
            chosen = None
            for value in order[k:]:
                if value > benchmark:
                    chosen = value
                    break
            if chosen is None:
                chosen = order[-1]

        successes += (chosen == best_value)
    return successes / reps

sim_prob = simulate_secretary(n=100, k=best_k, reps=20_000)
print("Simulated success probability at best k:", sim_prob)
print("Exact success probability at best k:", best_prob)

### Full Solution

If the best prize is in position $i>k$, the strategy succeeds only if the best among the first $i-1$ prizes is among the first $k$ rejected prizes.  
That probability is $k/(i-1)$. Since the position of the best prize is uniform over $\{1,\ldots,n\}$,

\[
P_k(\text{best})
=
\sum_{i=k+1}^n P(X=i)P(\text{success}\mid X=i)
=
\sum_{i=k+1}^n \frac1n\frac{k}{i-1}.
\]

For large $n$, this behaves like

\[
\frac{k}{n}\log\left(\frac{n}{k}\right),
\]

which is maximized near $k=n/e$, giving success probability near $1/e$.

## 4. Gambler's Ruin

A gambler starts with $k$ dollars and stops when reaching either $0$ dollars or $N$ dollars.

Each round:

\[
\text{win }1\text{ with probability }p,\qquad
\text{lose }1\text{ with probability }q=1-p.
\]

Let $u_k=P(\text{hit }N\text{ before }0\mid \text{start at }k)$.

The classical formula is

\[
u_k=
\begin{cases}
\dfrac{1-(q/p)^k}{1-(q/p)^N}, & p\ne q,\\[1.2em]
\dfrac{k}{N}, & p=q=\frac12.
\end{cases}
\]

In [ ]:
def gambler_ruin_prob(k, N, p):
    q = 1 - p
    if abs(p - q) < 1e-12:
        return k / N
    r = q / p
    return (1 - r**k) / (1 - r**N)

def simulate_gambler_ruin(k, N, p, reps=50_000):
    wins = 0
    for _ in range(reps):
        wealth = k
        while 0 < wealth < N:
            wealth += 1 if rng.random() < p else -1
        wins += (wealth == N)
    return wins / reps

N_total = 10
start_k = 4

rows = []
for p in [0.45, 0.50, 0.55]:
    theory = gambler_ruin_prob(start_k, N_total, p)
    sim = simulate_gambler_ruin(start_k, N_total, p, reps=20_000)
    rows.append([p, theory, sim])

pd.DataFrame(rows, columns=["p", "Theory P(hit N first)", "Simulation"]).style.format("{:.4f}")

### Full Solution

The hitting probability satisfies the recursion

\[
u_k=pu_{k+1}+qu_{k-1},\qquad u_0=0,\quad u_N=1.
\]

Solving this second-order difference equation gives

\[
u_k=
\frac{1-(q/p)^k}{1-(q/p)^N}
\]

when $p\ne q$. When $p=q=1/2$, the solution becomes linear:

\[
u_k=\frac{k}{N}.
\]

## 5. Memoryless Property of the Exponential Distribution

If $X\sim \text{Exponential}(\lambda)$, then

\[
P(X>s+t\mid X>s)=P(X>t).
\]

This is the memoryless property.

In [ ]:
lam = 2.0
N = 500_000
X = rng.exponential(scale=1/lam, size=N)

s = 0.7
t = 0.4

lhs = np.mean(X > s + t) / np.mean(X > s)
rhs_empirical = np.mean(X > t)
rhs_theory = np.exp(-lam * t)

print("Estimated P(X>s+t | X>s):", lhs)
print("Estimated P(X>t):", rhs_empirical)
print("Theoretical exp(-lambda t):", rhs_theory)

### Full Solution

For $X\sim \mathrm{Exp}(\lambda)$,

\[
P(X>s+t\mid X>s)
=
\frac{P(X>s+t)}{P(X>s)}
=
\frac{e^{-\lambda(s+t)}}{e^{-\lambda s}}
=
e^{-\lambda t}
=
P(X>t).
\]

## 6. Mixed Conditional Distributions

The lecture example has joint density/mass

\[
p_{X,Y}(x,y)=\frac{\lambda y^x e^{-(\lambda+1)y}}{x!},
\qquad x=0,1,2,\ldots,\quad y>0.
\]

This corresponds to the hierarchical model

\[
Y\sim \mathrm{Exponential}(\lambda),\qquad
X\mid Y=y\sim \mathrm{Poisson}(y).
\]

Then

\[
X\mid Y=y\sim \mathrm{Poisson}(y),
\]

and

\[
Y\mid X=x\sim \mathrm{Gamma}(x+1,\lambda+1)
\]

using the **rate** parameterization.

In [ ]:
lam = 1.5
N = 300_000

Y = rng.exponential(scale=1/lam, size=N)
X = rng.poisson(Y)

# Conditional distribution Y | X = x0
x0 = 3
Y_given_x0 = Y[X == x0]

posterior_shape = x0 + 1
posterior_rate = lam + 1
posterior_mean = posterior_shape / posterior_rate

print(f"Number of simulated observations with X={x0}:", len(Y_given_x0))
print("Empirical E[Y | X=3]:", Y_given_x0.mean())
print("Theoretical Gamma mean:", posterior_mean)

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(Y_given_x0, bins=40, density=True, alpha=0.6, label="Simulation: Y | X=3")

grid = np.linspace(0, np.quantile(Y_given_x0, 0.995), 300)
gamma_pdf = stats.gamma.pdf(grid, a=posterior_shape, scale=1/posterior_rate)
plt.plot(grid, gamma_pdf, label="Gamma(4, rate=2.5) theory")

plt.xlabel("y")
plt.ylabel("Density")
plt.title("Conditional Distribution Y | X=3")
plt.legend()
plt.show()

### Full Solution

From the joint expression,

\[
p_{X\mid Y}(x\mid y)
\propto
\frac{y^x}{x!},
\]

and the normalizing factor is $e^{-y}$, so

\[
X\mid Y=y\sim \mathrm{Poisson}(y).
\]

For $Y\mid X=x$, treat $x$ as fixed:

\[
p_{Y\mid X}(y\mid x)
\propto
y^x e^{-(\lambda+1)y}.
\]

This is a Gamma density with shape $x+1$ and rate $\lambda+1$.

## 7. Conditional Expectation as a Random Variable

The conditional expectation

\[
E[X\mid Y=y]
\]

is a number for each value $y$. The expression

\[
E[X\mid Y]
\]

is itself a random variable: it is a function of $Y$.

### Example: Unit Disk

Let $(X,Y)$ be uniformly distributed on the unit disk

\[
D=\{(x,y):x^2+y^2\le 1\}.
\]

Given $Y=y$,

\[
X\mid Y=y\sim \mathrm{Uniform}\left(-\sqrt{1-y^2},\sqrt{1-y^2}\right),
\]

so

\[
E[X\mid Y=y]=0.
\]

Therefore

\[
E[X\mid Y]=0.
\]

In [ ]:
# Simulate uniform points in the unit disk
N = 300_000
theta = rng.uniform(0, 2*np.pi, size=N)
R = np.sqrt(rng.uniform(0, 1, size=N))
X_disk = R * np.cos(theta)
Y_disk = R * np.sin(theta)

print("Estimated E[X]:", X_disk.mean())
print("Estimated E[Y]:", Y_disk.mean())
print("Estimated Cov(X,Y):", np.cov(X_disk, Y_disk, ddof=0)[0, 1])
print("Estimated Corr(X,Y):", np.corrcoef(X_disk, Y_disk)[0, 1])

In [ ]:
# Show non-independence: compare P(X>0.8 and Y>0.8) to product.
# In the unit disk, X>0.8 and Y>0.8 is impossible because 0.8^2+0.8^2>1.
event_x = X_disk > 0.8
event_y = Y_disk > 0.8

p_x = event_x.mean()
p_y = event_y.mean()
p_joint = (event_x & event_y).mean()

print("P(X>0.8):", p_x)
print("P(Y>0.8):", p_y)
print("P(X>0.8)P(Y>0.8):", p_x * p_y)
print("P(X>0.8, Y>0.8):", p_joint)

### Full Solution

The disk is symmetric left-to-right. For any fixed $Y=y$, the possible $X$ values are symmetric around zero. Therefore

\[
E[X\mid Y=y]=0.
\]

Thus

\[
E[XY]
=
E\{Y E[X\mid Y]\}
=
E[Y\cdot 0]
=
0.
\]

So $X$ and $Y$ are uncorrelated. But they are not independent: for example, $X>0.8$ and $Y>0.8$ cannot happen simultaneously in the unit disk.

## 8. Covariance Using Conditional Expectation

Suppose

\[
X\sim \mathrm{Uniform}(1,2),
\qquad
Y\mid X=x\sim \mathrm{Exponential}(\text{rate}=x).
\]

Then

\[
E[Y\mid X=x]=\frac{1}{x}.
\]

We want to compute

\[
\operatorname{Cov}(X,Y).
\]

In [ ]:
N = 300_000
X = rng.uniform(1, 2, size=N)
Y = rng.exponential(scale=1/X)

emp_cov = np.cov(X, Y, ddof=0)[0, 1]

theory_EX = 1.5
theory_EY = np.log(2)       # E[1/X] for X~Unif(1,2)
theory_EXY = 1.0            # E[X * E(Y|X)] = E[X*(1/X)] = 1
theory_cov = theory_EXY - theory_EX * theory_EY

print("Empirical Cov(X,Y):", emp_cov)
print("Theoretical Cov(X,Y):", theory_cov)

### Full Solution

Use iterated expectation:

\[
E[Y]=E[E(Y\mid X)]=E\left[\frac1X\right]
=\int_1^2 \frac1x\,dx=\log 2.
\]

Also,

\[
E[XY]=E[XE(Y\mid X)]
=
E\left[X\frac1X\right]
=
1.
\]

Since $E[X]=3/2$,

\[
\operatorname{Cov}(X,Y)
=
E[XY]-E[X]E[Y]
=
1-\frac32\log 2.
\]

## 9. Law of Total Expectation

The law of total expectation says

\[
E[X]=E[E(X\mid Y)].
\]

More generally,

\[
E[g(X,Y)]=E\big[E(g(X,Y)\mid X)\big].
\]

We illustrate this with a mixture-binomial model.

In [ ]:
# Model: U ~ Uniform(0,1), S | U=u ~ Binomial(n,u)
n = 20
N = 300_000

U = rng.uniform(0, 1, size=N)
S = rng.binomial(n, U)

emp_mean = S.mean()
theory_mean = n * 0.5

print("Empirical E[S]:", emp_mean)
print("Theoretical E[E(S|U)] = E[nU] = n/2:", theory_mean)

### Full Solution

Since

\[
S\mid U=u\sim \mathrm{Binomial}(n,u),
\]

we have

\[
E[S\mid U=u]=nu.
\]

Therefore,

\[
E[S]=E[E(S\mid U)]
=
E[nU]
=
nE[U]
=
\frac n2.
\]

## 10. Law of Total Variance

The law of total variance says

\[
\operatorname{Var}(Y)=E[\operatorname{Var}(Y\mid X)]
+
\operatorname{Var}(E[Y\mid X]).
\]

For the model

\[
U\sim \mathrm{Uniform}(0,1),\qquad S\mid U=u\sim \mathrm{Binomial}(n,u),
\]

we have

\[
E[S\mid U]=nU,\qquad
\operatorname{Var}(S\mid U)=nU(1-U).
\]

In [ ]:
emp_var = S.var(ddof=0)

theory_var = n * (1/2 - 1/3) + n**2 * (1/12)

print("Empirical Var(S):", emp_var)
print("Theoretical Var(S):", theory_var)

components = pd.DataFrame({
    "Component": ["E[Var(S|U)]", "Var(E[S|U])", "Total"],
    "Value": [n*(1/2 - 1/3), n**2*(1/12), theory_var]
})
display(components)

### Full Solution

Compute

\[
E[\operatorname{Var}(S\mid U)]
=
E[nU(1-U)]
=
n\left(E[U]-E[U^2]\right)
=
n\left(\frac12-\frac13\right)
=
\frac n6.
\]

Also,

\[
\operatorname{Var}(E[S\mid U])
=
\operatorname{Var}(nU)
=
n^2\operatorname{Var}(U)
=
\frac{n^2}{12}.
\]

Hence

\[
\operatorname{Var}(S)=\frac n6+\frac{n^2}{12}.
\]

## 11. Random Sums

Let

\[
Y=X_1+\cdots+X_N,
\]

where $N$ is a nonnegative integer-valued random variable independent of the i.i.d. variables $X_i$.

If

\[
E[X_i]=\mu,\qquad \operatorname{Var}(X_i)=\sigma^2,
\]

then

\[
E[Y]=E[N]\mu.
\]

Also,

\[
\operatorname{Var}(Y)=E[N]\sigma^2+\operatorname{Var}(N)\mu^2.
\]

We verify this by simulation.

In [ ]:
# Example: N ~ Poisson(lambda_N), X_i ~ Exponential(mean theta)
lambda_N = 4.0
theta = 2.0
Nsim = 200_000

Ns = rng.poisson(lambda_N, size=Nsim)
Ys = np.zeros(Nsim)

for i, ni in enumerate(Ns):
    if ni > 0:
        Ys[i] = rng.exponential(scale=theta, size=ni).sum()

emp_mean = Ys.mean()
emp_var = Ys.var(ddof=0)

mu = theta
sigma2 = theta**2
theory_mean = lambda_N * mu
theory_var = lambda_N * sigma2 + lambda_N * mu**2

print("Empirical E[Y]:", emp_mean)
print("Theoretical E[Y]:", theory_mean)
print("Empirical Var(Y):", emp_var)
print("Theoretical Var(Y):", theory_var)

### Full Solution

Condition on $N$:

\[
E[Y\mid N=n]=E[X_1+\cdots+X_n]=n\mu.
\]

Therefore,

\[
E[Y]=E[E(Y\mid N)]=E[N\mu]=\mu E[N].
\]

For the variance,

\[
\operatorname{Var}(Y\mid N=n)=n\sigma^2.
\]

Thus

\[
E[\operatorname{Var}(Y\mid N)]=\sigma^2E[N],
\]

and

\[
\operatorname{Var}(E[Y\mid N])
=
\operatorname{Var}(N\mu)
=
\mu^2\operatorname{Var}(N).
\]

Combining gives

\[
\operatorname{Var}(Y)=E[N]\sigma^2+\operatorname{Var}(N)\mu^2.
\]

## 12. Covariance Projection Identity

The lecture states the identity

\[
\operatorname{Cov}(g(X),h(Y))
=
\operatorname{Cov}(g(X),E[h(Y)\mid X]).
\]

We verify this numerically for the model

\[
X\sim \mathrm{Uniform}(1,2),\qquad
Y\mid X=x\sim \mathrm{Exponential}(\text{rate}=x).
\]

Take $g(X)=X^2$ and $h(Y)=Y$.

In [ ]:
N = 400_000
X = rng.uniform(1, 2, size=N)
Y = rng.exponential(scale=1/X)

gX = X**2
hY = Y
EhY_given_X = 1/X

left = np.cov(gX, hY, ddof=0)[0, 1]
right = np.cov(gX, EhY_given_X, ddof=0)[0, 1]

print("Cov(g(X), h(Y)):", left)
print("Cov(g(X), E[h(Y)|X]):", right)
print("Difference:", left - right)

### Full Solution

Starting from the definition,

\[
\operatorname{Cov}(g(X),h(Y))
=
E[g(X)h(Y)]-E[g(X)]E[h(Y)].
\]

Now condition on $X$:

\[
E[g(X)h(Y)]
=
E\left[E(g(X)h(Y)\mid X)\right]
=
E\left[g(X)E(h(Y)\mid X)\right].
\]

Also,

\[
E[h(Y)]=E[E(h(Y)\mid X)].
\]

Therefore,

\[
\operatorname{Cov}(g(X),h(Y))
=
\operatorname{Cov}(g(X),E[h(Y)\mid X]).
\]

## 13. Bayesian Updating: Beta-Binomial Model

Let

\[
p\sim \mathrm{Beta}(\alpha,\beta),
\qquad
X\mid p\sim \mathrm{Binomial}(n,p).
\]

Then the posterior distribution is

\[
p\mid X=x \sim \mathrm{Beta}(\alpha+x,\beta+n-x).
\]

A special case in the lecture is the uniform prior $p\sim \mathrm{Beta}(1,1)$.

In [ ]:
alpha_prior = 1
beta_prior = 1
n = 20
x_obs = 14

alpha_post = alpha_prior + x_obs
beta_post = beta_prior + n - x_obs

posterior_mean = alpha_post / (alpha_post + beta_post)
posterior_var = (alpha_post * beta_post) / (((alpha_post + beta_post)**2) * (alpha_post + beta_post + 1))

print("Posterior distribution: Beta({}, {})".format(alpha_post, beta_post))
print("Posterior mean:", posterior_mean)
print("Posterior variance:", posterior_var)

In [ ]:
grid = np.linspace(0, 1, 400)
prior_pdf = stats.beta.pdf(grid, alpha_prior, beta_prior)
posterior_pdf = stats.beta.pdf(grid, alpha_post, beta_post)

plt.figure(figsize=(7, 4))
plt.plot(grid, prior_pdf, label="Prior Beta(1,1)")
plt.plot(grid, posterior_pdf, label=f"Posterior Beta({alpha_post},{beta_post})")
plt.axvline(x_obs/n, linestyle="--", label="sample proportion")
plt.xlabel("p")
plt.ylabel("Density")
plt.title("Beta-Binomial Bayesian Updating")
plt.legend()
plt.show()

### Full Solution

The likelihood is

\[
P(X=x\mid p)=\binom{n}{x}p^x(1-p)^{n-x}.
\]

The prior density is

\[
\pi(p)\propto p^{\alpha-1}(1-p)^{\beta-1}.
\]

Thus the posterior is proportional to

\[
p^x(1-p)^{n-x}p^{\alpha-1}(1-p)^{\beta-1}
=
p^{\alpha+x-1}(1-p)^{\beta+n-x-1}.
\]

So

\[
p\mid X=x\sim \mathrm{Beta}(\alpha+x,\beta+n-x).
\]

## 14. Missing Data and Inverse Probability Weighting

Suppose we want

\[
\mu=E[Y],
\]

but sometimes $Y$ is missing. Let $R=1$ if $Y$ is observed and $R=0$ otherwise.

Assume missing at random:

\[
P(R=1\mid X,Y)=P(R=1\mid X)=\pi(X).
\]

The inverse probability weighting quantity is

\[
W=\frac{RY}{\pi(X)}.
\]

Then

\[
E[W]=E[Y].
\]

In [ ]:
N = 200_000

# Covariate X: age-like standardized variable
X = rng.uniform(0, 1, size=N)

# Outcome Y depends on X
Y = 40_000 + 30_000*X + rng.normal(0, 5_000, size=N)

# Response probability depends on X; known pi(X)
pi_X = 0.25 + 0.65*X
R = rng.binomial(1, pi_X)

true_mu = Y.mean()
observed_naive_mean = Y[R == 1].mean()
ipw_estimate = np.mean(R * Y / pi_X)

print("True mean E[Y] from full simulated population:", true_mu)
print("Naive complete-case mean:", observed_naive_mean)
print("IPW estimate:", ipw_estimate)
print("Response rate:", R.mean())

### Full Solution

Using iterated expectation,

\[
E\left[\frac{RY}{\pi(X)}\right]
=
E\left[
E\left(\frac{RY}{\pi(X)}\mid X,Y\right)
\right].
\]

Since $\pi(X)$ is known given $X$,

\[
E\left(\frac{RY}{\pi(X)}\mid X,Y\right)
=
\frac{Y}{\pi(X)}E(R\mid X,Y).
\]

By missing at random,

\[
E(R\mid X,Y)=P(R=1\mid X,Y)=\pi(X).
\]

Therefore,

\[
E\left(\frac{RY}{\pi(X)}\mid X,Y\right)=Y.
\]

Taking expectation again,

\[
E\left[\frac{RY}{\pi(X)}\right]=E[Y].
\]

So the IPW estimator is

\[
\hat\mu_{\mathrm{IPW}}
=
\frac1n\sum_{i=1}^n \frac{R_iY_i}{\pi(X_i)}.
\]

## 15. Survey Sampling with Districts

Suppose a city has three districts: A, B, and C.  
We want to estimate the city-wide average income.

If the district population proportions are known, then

\[
E[Y]=\sum_{d} E[Y\mid D=d]P(D=d).
\]

This is another use of the law of total expectation.

In [ ]:
districts = np.array(["A", "B", "C"])
population_weights = np.array([0.50, 0.30, 0.20])
district_means = np.array([55_000, 70_000, 90_000])
district_sds = np.array([8_000, 10_000, 12_000])

true_city_mean = np.sum(population_weights * district_means)
print("True city mean by law of total expectation:", true_city_mean)

# Suppose the survey oversamples district C, so sample proportions differ from population proportions.
sample_weights = np.array([0.30, 0.30, 0.40])
sample_n = 900

sample_district = rng.choice(districts, size=sample_n, p=sample_weights)

income = np.empty(sample_n)
for d, mu, sd in zip(districts, district_means, district_sds):
    mask = sample_district == d
    income[mask] = rng.normal(mu, sd, size=mask.sum())

survey = pd.DataFrame({"district": sample_district, "income": income})
display(survey.groupby("district")["income"].agg(["count", "mean"]))

naive = survey["income"].mean()

# Weighted estimator: sum population weight * sample mean within district
group_means = survey.groupby("district")["income"].mean().reindex(districts)
weighted = np.sum(population_weights * group_means.values)

print("Naive survey mean:", naive)
print("Weighted district mean:", weighted)
print("True city mean:", true_city_mean)

### Full Solution

The naive sample mean estimates the mean of the **sample design**, not necessarily the population.

If district C is oversampled, then the naive mean may overestimate the city average.  
Using the law of total expectation,

\[
E[Y]
=
E[E(Y\mid D)]
=
\sum_d E[Y\mid D=d]P(D=d).
\]

Therefore, estimate each district mean from the data and then combine using the known city population weights:

\[
\hat\mu
=
\sum_d \hat\mu_d w_d.
\]

# Practice Problems with Full Solutions

## Practice Problem 1 — Law of Total Expectation

Let $U\sim \mathrm{Uniform}(0,1)$ and, conditional on $U=u$,

\[
X\mid U=u\sim \mathrm{Poisson}(3u).
\]

Find $E[X]$.

In [ ]:
# Simulation solution
N = 300_000
U = rng.uniform(0, 1, size=N)
X = rng.poisson(3 * U)

print("Simulation estimate of E[X]:", X.mean())
print("Theory:", 1.5)

### Solution

Since

\[
E[X\mid U=u]=3u,
\]

we have

\[
E[X]=E[E(X\mid U)]=E[3U]=3E[U]=3\cdot\frac12=\frac32.
\]

## Practice Problem 2 — Law of Total Variance

Use the same model:

\[
U\sim \mathrm{Uniform}(0,1),
\qquad X\mid U=u\sim \mathrm{Poisson}(3u).
\]

Find $\operatorname{Var}(X)$.

In [ ]:
empirical_var = X.var(ddof=0)

# For Poisson: Var(X|U)=3U and E(X|U)=3U
theory_var = 3 * 0.5 + 9 * (1/12)

print("Simulation estimate of Var(X):", empirical_var)
print("Theory:", theory_var)

### Solution

For a Poisson random variable,

\[
\operatorname{Var}(X\mid U=u)=3u.
\]

Thus

\[
E[\operatorname{Var}(X\mid U)]=E[3U]=\frac32.
\]

Also,

\[
E[X\mid U]=3U,
\]

so

\[
\operatorname{Var}(E[X\mid U])
=
\operatorname{Var}(3U)
=
9\operatorname{Var}(U)
=
9\cdot\frac1{12}
=
\frac34.
\]

Therefore,

\[
\operatorname{Var}(X)
=
\frac32+\frac34
=
\frac94.
\]

## Practice Problem 3 — Random Sum

Let

\[
N\sim \mathrm{Poisson}(5),
\qquad X_i\sim \mathrm{Bernoulli}(0.4),
\]

independent. Define

\[
Y=X_1+\cdots+X_N.
\]

Find $E[Y]$ and $\operatorname{Var}(Y)$.

In [ ]:
Nsim = 300_000
Ns = rng.poisson(5, size=Nsim)
Ys = np.zeros(Nsim)

for i, ni in enumerate(Ns):
    if ni > 0:
        Ys[i] = rng.binomial(1, 0.4, size=ni).sum()

print("Simulation E[Y]:", Ys.mean())
print("Simulation Var(Y):", Ys.var(ddof=0))

mu = 0.4
sigma2 = 0.4 * 0.6
EN = 5
VarN = 5

print("Theory E[Y]:", EN * mu)
print("Theory Var(Y):", EN * sigma2 + VarN * mu**2)

### Solution

Here

\[
\mu=E[X_i]=0.4,
\qquad
\sigma^2=\operatorname{Var}(X_i)=0.4(0.6)=0.24.
\]

Since $N\sim \mathrm{Poisson}(5)$,

\[
E[N]=5,\qquad \operatorname{Var}(N)=5.
\]

Therefore,

\[
E[Y]=E[N]\mu=5(0.4)=2.
\]

Also,

\[
\operatorname{Var}(Y)
=
E[N]\sigma^2+\operatorname{Var}(N)\mu^2
=
5(0.24)+5(0.4)^2
=
1.2+0.8
=
2.
\]

This is consistent with Poisson thinning: $Y\sim \mathrm{Poisson}(5\cdot0.4)=\mathrm{Poisson}(2)$.

## Practice Problem 4 — Bayesian Updating

Suppose

\[
p\sim \mathrm{Beta}(2,3),
\qquad X\mid p\sim \mathrm{Binomial}(30,p).
\]

If $X=18$, find the posterior distribution and posterior mean.

In [ ]:
alpha, beta = 2, 3
n, x = 30, 18

alpha_post = alpha + x
beta_post = beta + n - x

post_mean = alpha_post / (alpha_post + beta_post)

print(f"Posterior: Beta({alpha_post}, {beta_post})")
print("Posterior mean:", post_mean)

### Solution

By Beta-Binomial conjugacy,

\[
p\mid X=x\sim \mathrm{Beta}(\alpha+x,\beta+n-x).
\]

Here,

\[
\alpha+x=2+18=20,
\qquad
\beta+n-x=3+30-18=15.
\]

So

\[
p\mid X=18\sim \mathrm{Beta}(20,15).
\]

The posterior mean is

\[
E[p\mid X=18]
=
\frac{20}{20+15}
=
\frac{20}{35}
=
0.5714.
\]

## Practice Problem 5 — IPW Estimator

Suppose $X\sim \mathrm{Uniform}(0,1)$,

\[
Y=10+5X+\epsilon,\qquad E[\epsilon]=0,
\]

and

\[
P(R=1\mid X,Y)=\pi(X)=0.5+0.4X.
\]

Show that

\[
E\left[\frac{RY}{\pi(X)}\right]=E[Y].
\]

Then verify by simulation.

In [ ]:
N = 300_000
X = rng.uniform(0, 1, size=N)
eps = rng.normal(0, 1, size=N)
Y = 10 + 5*X + eps

pi_X = 0.5 + 0.4*X
R = rng.binomial(1, pi_X)

true_mean = Y.mean()
ipw = np.mean(R * Y / pi_X)
naive = Y[R == 1].mean()

print("True E[Y] estimate:", true_mean)
print("IPW estimate:", ipw)
print("Naive observed-only estimate:", naive)

### Solution

Use conditional expectation:

\[
E\left[\frac{RY}{\pi(X)}\right]
=
E\left[
E\left(\frac{RY}{\pi(X)}\mid X,Y\right)
\right].
\]

Since $Y$ and $\pi(X)$ are known after conditioning on $(X,Y)$,

\[
E\left(\frac{RY}{\pi(X)}\mid X,Y\right)
=
\frac{Y}{\pi(X)}E(R\mid X,Y).
\]

The missing-at-random condition gives

\[
E(R\mid X,Y)=P(R=1\mid X,Y)=\pi(X).
\]

Therefore,

\[
E\left(\frac{RY}{\pi(X)}\mid X,Y\right)=Y,
\]

and hence

\[
E\left[\frac{RY}{\pi(X)}\right]=E[Y].
\]

# Summary

In this lab, we used computation to verify and apply the main ideas of Section 6:

- Conditional PMFs/PDFs
- Law of total probability
- Conditioning on continuous random variables
- Best-prize problem
- Gambler's ruin
- Exponential memoryless property
- Mixed conditional distributions
- Conditional expectation as a random variable
- Law of total expectation
- Law of total variance
- Random sums
- Covariance identity
- Bayesian updating
- Missing data and inverse probability weighting
- Survey sampling through conditioning

The central message is:

\[
\boxed{\text{Conditioning simplifies hard probability problems.}}
\]